In [20]:
import os
import json
from scraper import fetch_website_contents,fetch_website_links
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display,Markdown,update_display

In [21]:
load_dotenv(override=True)

google_base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
google_api_key=os.getenv("GOOGLE_API_KEY")

gemini_client=OpenAI(base_url=google_base_url,api_key=google_api_key)


In [22]:
all_links=fetch_website_links("https://www.nvidia.com")
all_links

['https://www.nvidia.com',
 '#page-content',
 'https://www.nvidia.com/en-in/',
 'https://www.nvidia.com/en-in/industries/healthcare-life-sciences/biopharma/',
 'https://www.nvidia.com/en-in/data-center/dgx-cloud/',
 'https://build.nvidia.com/',
 'https://docs.nvidia.com/ngc/latest/ngc-private-registry-user-guide.html',
 'https://www.nvidia.com/en-in/gpu-cloud/',
 'https://www.nvidia.com/en-in/studio/',
 'https://www.nvidia.com/en-in/geforce/broadcasting/',
 'https://www.nvidia.com/en-in/software/nvidia-app/',
 'https://www.nvidia.com/en-in/ai-on-rtx/',
 'https://www.nvidia.com/en-in/geforce/rtx-remix/',
 'https://www.nvidia.com/en-in/software/nvidia-app/g-assist/',
 'https://www.nvidia.com/en-in/data-center/',
 'https://www.nvidia.com/en-in/data-center/dgx-platform/',
 'https://www.nvidia.com/en-in/data-center/grace-cpu/',
 'https://www.nvidia.com/en-in/data-center/hgx/',
 'https://www.nvidia.com/en-in/edge-computing/products/igx/',
 'https://www.nvidia.com/en-in/data-center/products/m

In [23]:
system_prompt="""you are a helpful,analytic,resourcful assitant who is excellent at finding relevent links 
from wesbite.The website is about some comapny and you have to give the links that are relevent to the comapines
about us,the services they provide and the main idea of the company.
Don't add any links to content page,email or addresses or pdfs or anything else.i need only the links that can go to another webpage.
You give the output in a json format which is as follows: 

{
    links:[
    {"type":"about us","url":"htts://www.example.com/about"},
    {"type":"services","url":"htts://www.example.com/services"}
    ]
}  """

In [24]:
user_prompt=f"""i want you to choose the relevent links from the website
that talk about the companies about,its services and people.
Here are are all the links:\n"""

user_prompt+="\n".join(all_links)

In [25]:
def get_msgs():
    return [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}]

In [26]:
def getrellinks():
    response=gemini_client.chat.completions.create(model="gemini-3.1-flash-lite",messages=get_msgs(),
    response_format={"type":"json_object"})
    result=response.choices[0].message.content
    rel_links=json.loads(result)
    return rel_links

In [27]:
getrellinks()

{'links': [{'type': 'about us',
   'url': 'https://www.nvidia.com/en-in/about-nvidia/careers/'},
  {'type': 'about us', 'url': 'https://nvidianews.nvidia.com/'},
  {'type': 'about us', 'url': 'https://investor.nvidia.com/home/default.aspx'},
  {'type': 'about us',
   'url': 'https://www.nvidia.com/en-in/executive-insights/'},
  {'type': 'services', 'url': 'https://www.nvidia.com/en-in/data-center/'},
  {'type': 'services', 'url': 'https://www.nvidia.com/en-in/geforce/'},
  {'type': 'services', 'url': 'https://www.nvidia.com/en-in/omniverse/'},
  {'type': 'services', 'url': 'https://www.nvidia.com/en-in/ai/'},
  {'type': 'services', 'url': 'https://www.nvidia.com/en-in/networking/'},
  {'type': 'services',
   'url': 'https://www.nvidia.com/en-in/self-driving-cars/'},
  {'type': 'main idea',
   'url': 'https://www.nvidia.com/en-in/about-nvidia/ai-for-good/'},
  {'type': 'main idea', 'url': 'https://www.nvidia.com/en-us/research/'},
  {'type': 'main idea',
   'url': 'https://www.nvidia.co

In [28]:
def getcontent():
    result=fetch_website_contents("https://www.nvidia.com")
    rel_links=getrellinks()
    results=f"the content is {result}\n\n"
    for links in rel_links["links"]:
        content=fetch_website_contents(links["url"])
        results+=content
    return results

In [29]:
system_prompt_content=f"""you are a helpful assistant who's given many selected links from the wessite 
of a company and you job to get the content 
out of these links in an analytical",
useful way to create a comapany brochure that can 
be presented to the public."""

In [30]:
user_prompt_content=f"""im giving you the content of a company wesbite and i want you to use it s create
a comapany brochure for me tnat can be presened to the public.the content is as follows:\n\n{getcontent()}"""

In [31]:
def get_msgs_content():
    return [{"role":"system","content":system_prompt_content},
            {"role":"user","content":user_prompt_content}]

In [ ]:
#simple output in Markdown format
def brochure_generator():
    response=gemini_client.chat.completions.create(model="gemini-3.1-flash-lite",
                                                   messages=get_msgs_content())
    results = response.choices[0].message.content
    return display(Markdown(results))

In [40]:
#adding a function to get the output in a stream.
def stream_brochure():
    stream=gemini_client.chat.completions.create(model="gemini-3.1-flash-lite",
                                                   messages=get_msgs_content(),
                                                   stream=True)
    response=""
    display_handle=display(Markdown(""),display_id=True)
    for chunk in stream:
        response+=chunk.choices[0].delta.content or ''
        update_display(Markdown(response),display_id=display_handle.display_id)
    return response

In [ ]:
stream_brochure()